# Example 1 -- Data Schema Inspection

An AI agent can learn the *structure* of a dataset (column names, dtypes, null counts, unique value counts, numeric summaries) **without ever seeing individual rows**.

**Use case:** An agent needs to understand what data is available before writing analysis code, but must not be exposed to raw records.

## 1. Prepare a Sample Dataset

In production this would be loaded by a trusted data owner, never by the agent itself.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from agent_privacy_layer import PrivacyLayer, UserConfirmation

df = pd.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie", "Diana", "Eve"],
        "email": [
            "alice@example.com",
            "bob@test.org",
            "charlie@mail.com",
            "diana@example.com",
            "eve@test.org",
        ],
        "age": [25, 30, 28, 35, 22],
        "salary": [50_000, 60_000, 55_000, 70_000, 45_000],
        "department": ["Engineering", "Sales", "Engineering", "Sales", "Engineering"],
    }
)

print(f"Dataset created with {len(df)} rows and {len(df.columns)} columns")

Dataset created with 5 rows and 5 columns


## 2. Create the Privacy Layer

`dry_run=True` auto-approves all prompts (for demonstration purposes).

In [2]:
layer = PrivacyLayer(
    df,
    epsilon=1.0,
    confirmation=UserConfirmation(dry_run=True),
)

## 3. Inspect the Schema

The agent can safely inspect the schema -- no raw rows are returned.

In [3]:
schema = layer.inspect_data()
print("=== Full schema ===")
print(schema)

=== Full schema ===
Rows: 5  Columns: 5

  'name'                         dtype=str             nulls=0  unique=5  sample_values=[Alice, Bob, Charlie, Diana, Eve]
  'email'                        dtype=str             nulls=0  unique=5  sample_values=[alice@example.com, bob@test.org, charlie@mail.com, diana@example.com, eve@test.org]
  'age'                          dtype=int64           nulls=0  unique=5  sample_values=[25, 30, 28, 35, 22]
  'salary'                       dtype=int64           nulls=0  unique=5  sample_values=[50000, 60000, 55000, 70000, 45000]
  'department'                   dtype=str             nulls=0  unique=2  sample_values=[Engineering, Sales]


In [4]:
# Individual accessors
print("Column names:", layer.column_names())
print("Dtypes:      ", layer.dtypes())
print("Shape:       ", layer.shape())

Column names: ['name', 'email', 'age', 'salary', 'department']
Dtypes:       {'name': 'str', 'email': 'str', 'age': 'int64', 'salary': 'int64', 'department': 'str'}
Shape:        (5, 5)


## 4. Numeric Summaries

Aggregate statistics, not individual values.

In [5]:
print("=== Numeric summary for 'age' ===")
age_summary = layer.numeric_summary("age")
for stat, value in age_summary.items():
    print(f"  {stat:>7s}: {value:.2f}")

print()
print("=== Numeric summary for 'salary' ===")
salary_summary = layer.numeric_summary("salary")
for stat, value in salary_summary.items():
    print(f"  {stat:>7s}: {value:.2f}")

=== Numeric summary for 'age' ===
      min: 22.00
      max: 35.00
     mean: 28.00
      std: 4.95
   median: 28.00
      q25: 25.00
      q75: 30.00

=== Numeric summary for 'salary' ===
      min: 45000.00
      max: 70000.00
     mean: 56000.00
      std: 9617.69
   median: 55000.00
      q25: 50000.00
      q75: 60000.00


## Analysis

**Goal:** Allow an AI agent to understand dataset structure without exposing individual rows.

**Results:**
- **Schema inspection** correctly reveals column names, data types, null counts, and unique value counts -- all metadata that does not leak individual records.
- **Individual accessors** (`column_names()`, `dtypes()`, `shape()`) provide quick structural summaries suitable for agent planning.
- **Numeric summaries** return aggregate statistics (min, max, mean, std, median, quartiles) for numeric columns. These are safe to expose because they describe the distribution, not individual values.

**Verdict:** The data inspection API successfully achieves its goal. An agent can learn that the dataset has 5 rows and 5 columns, that ages range from 22-35 with a mean of 28, and that salaries range from 45k-70k with a mean of 56k -- all without seeing a single raw record. The `sample_values` field in the full schema does show actual values, which is an intentional design choice for categorical/string columns to help the agent understand the domain.